# PCD Assignment 01 — Image Down-Sampling & Up-Sampling
**Mata Kuliah:** Pengolahan Citra Digital (Digital Image Processing)
**Semester:** 3 — Ilmu Komputer
**Topik:** Down Sampling (Max, Average, Median) & Up Sampling (Nearest Neighbor, Bilinear, Bicubic)

Notebook ini mengimplementasikan dan membandingkan:

1. **Down Sampling** — mengurangi resolusi citra menggunakan 3 metode pooling: **Max**, **Average**, dan **Median**.
2. **Up Sampling** — memperbesar kembali citra menggunakan 3 metode interpolasi: **Nearest Neighbor (NN)**, **Bilinear**, dan **Bicubic**.

Tiga citra uji dengan karakteristik berbeda digunakan:
- **Astronaut** — citra berwarna (RGB) dengan tekstur halus, kulit, dan tepi objek yang jelas.
- **Cameraman** — citra grayscale klasik dengan detail halus dan gradasi kontras.
- **Checkerboard** — pola frekuensi tinggi (kotak-kotak berulang), ideal untuk mengamati efek *aliasing*.

Jalankan seluruh sel secara berurutan (Runtime → Run all) di Google Colab.


## 1. Setup & Instalasi Library

In [ ]:
!pip -q install scikit-image opencv-python-headless matplotlib numpy
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
import skimage.data as data
import os

plt.rcParams['figure.dpi'] = 110
np.random.seed(0)


## 2. Menyiapkan Citra Uji
Kita menggunakan tiga citra dengan karakteristik berbeda. Jika Anda ingin menggunakan
citra sendiri, upload file ke Colab (folder `/content/`) lalu ganti path pada dictionary
`images` di bawah ini dengan `cv2.imread('/content/nama_file.jpg')`.


In [ ]:
os.makedirs('images', exist_ok=True)

# Citra bawaan scikit-image (tidak perlu koneksi internet tambahan)
astronaut = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)          # RGB, tekstur halus
camera    = cv2.cvtColor(data.camera(), cv2.COLOR_GRAY2BGR)            # Grayscale, detail halus
checker   = cv2.cvtColor(data.checkerboard(), cv2.COLOR_GRAY2BGR)      # Pola frekuensi tinggi

cv2.imwrite('images/astronaut.png', astronaut)
cv2.imwrite('images/camera.png', camera)
cv2.imwrite('images/checkerboard.png', checker)

images = {
    'astronaut': astronaut,
    'camera': camera,
    'checkerboard': checker,
}

def to_rgb(im):
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, im) in zip(axes, images.items()):
    ax.imshow(to_rgb(im))
    ax.set_title(f'{name}\n{im.shape[1]}x{im.shape[0]}')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 3. Implementasi Down Sampling (Max, Average, Median Pooling)

Down sampling dilakukan dengan cara membagi citra menjadi blok-blok berukuran
`factor x factor` piksel, lalu setiap blok direduksi menjadi satu piksel
menggunakan salah satu operasi statistik berikut:

- **Max Pooling** — mengambil nilai piksel **maksimum** dalam tiap blok. Cenderung
  mempertahankan bagian terang/tepi tajam, namun dapat memunculkan bintik terang berlebih (noise amplification).
- **Average Pooling** — mengambil **rata-rata** nilai piksel dalam tiap blok. Menghasilkan
  citra yang lebih halus, meredam noise, tetapi dapat mengaburkan tepi.
- **Median Pooling** — mengambil nilai **tengah (median)** dari tiap blok. Baik untuk
  meredam noise impulsif (salt-and-pepper) tanpa mengaburkan tepi sebanyak average pooling.


In [ ]:
def downsample(img, factor, method='average'):
    """Down-sample citra dengan block pooling.
    img    : array HxW atau HxWxC (uint8)
    factor : faktor downsampling (integer), misal 4 -> blok 4x4 jadi 1 piksel
    method : 'max' | 'average' | 'median'
    """
    h, w = img.shape[:2]
    h2, w2 = h - (h % factor), w - (w % factor)   # crop agar habis dibagi
    img = img[:h2, :w2]

    if img.ndim == 2:
        img = img[:, :, None]
    H, W, C = img.shape

    # Susun ulang jadi blok berukuran factor x factor
    blocks = img.reshape(H // factor, factor, W // factor, factor, C)
    blocks = blocks.transpose(0, 2, 1, 3, 4).reshape(H // factor, W // factor, factor * factor, C)

    if method == 'max':
        out = blocks.max(axis=2)
    elif method == 'average':
        out = blocks.mean(axis=2)
    elif method == 'median':
        out = np.median(blocks, axis=2)
    else:
        raise ValueError('Metode tidak dikenal: gunakan max/average/median')

    out = np.clip(out, 0, 255).astype(np.uint8)
    if out.shape[2] == 1:
        out = out[:, :, 0]
    return out


## 4. Implementasi Up Sampling (Nearest Neighbor, Bilinear, Bicubic)

Up sampling memperbesar kembali citra kecil ke ukuran semula menggunakan tiga
metode interpolasi (menggunakan `cv2.resize`):

- **Nearest Neighbor (NN)** — setiap piksel baru mengambil nilai piksel tetangga
  terdekat. Paling cepat, tetapi menghasilkan efek **blocky / kotak-kotak** (blocky artifact).
- **Bilinear** — nilai piksel baru dihitung dari interpolasi linear 4 piksel tetangga
  terdekat (2x2). Lebih halus dari NN, tetapi bisa sedikit mengaburkan tepi tajam.
- **Bicubic** — menggunakan interpolasi kubik dari 16 piksel tetangga (4x4). Hasil
  paling halus dan tajam di antara ketiganya, namun komputasinya lebih berat.


In [ ]:
def upsample(img, factor, method='bilinear'):
    """Up-sample citra menggunakan interpolasi.
    method : 'nn' | 'bilinear' | 'bicubic'
    """
    h, w = img.shape[:2]
    new_size = (w * factor, h * factor)
    interp_map = {
        'nn': cv2.INTER_NEAREST,
        'bilinear': cv2.INTER_LINEAR,
        'bicubic': cv2.INTER_CUBIC,
    }
    return cv2.resize(img, new_size, interpolation=interp_map[method])


## 5. Visualisasi: Perbandingan Metode Down Sampling

In [ ]:
FACTOR = 8

fig, axes = plt.subplots(len(images), 4, figsize=(14, 4 * len(images)))
for row, (name, img) in enumerate(images.items()):
    axes[row, 0].imshow(to_rgb(img))
    axes[row, 0].set_title(f'{name} — Original\n{img.shape[1]}x{img.shape[0]}')
    axes[row, 0].axis('off')
    for col, method in enumerate(['max', 'average', 'median'], start=1):
        d = downsample(img, FACTOR, method)
        axes[row, col].imshow(to_rgb(d))
        axes[row, col].set_title(f'{method.capitalize()} pool\n{d.shape[1]}x{d.shape[0]}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig('downsampling_comparison.png')
plt.show()


## 6. Visualisasi: Perbandingan Metode Up Sampling

Setiap citra terlebih dahulu di-*downsample* (Average pooling, factor 8) lalu di-*upsample* kembali ke ukuran semula dengan tiga metode interpolasi.

In [ ]:
fig, axes = plt.subplots(len(images), 4, figsize=(14, 4 * len(images)))
for row, (name, img) in enumerate(images.items()):
    small = downsample(img, FACTOR, 'average')
    axes[row, 0].imshow(to_rgb(small))
    axes[row, 0].set_title(f'{name} — Small input\n{small.shape[1]}x{small.shape[0]}')
    axes[row, 0].axis('off')
    for col, method in enumerate(['nn', 'bilinear', 'bicubic'], start=1):
        u = upsample(small, FACTOR, method)
        axes[row, col].imshow(to_rgb(u))
        axes[row, col].set_title(f'{method.upper() if method=="nn" else method.capitalize()} up\n{u.shape[1]}x{u.shape[0]}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig('upsampling_comparison.png')
plt.show()


## 7. Zoom-in: Detail Tekstur Hasil Up Sampling

Untuk melihat perbedaan lebih jelas, kita perbesar (crop) bagian wajah pada citra *astronaut*
hasil up sampling dari masing-masing metode.


In [ ]:
def crop_center(im, cy, cx, s):
    return im[cy - s:cy + s, cx - s:cx + s]

img = images['astronaut']
small = downsample(img, FACTOR, 'average')

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, method in zip(axes, ['nn', 'bilinear', 'bicubic']):
    u = upsample(small, FACTOR, method)
    c = crop_center(u, 200, 200, 80)
    ax.imshow(to_rgb(c))
    ax.set_title(method)
    ax.axis('off')
plt.tight_layout()
plt.savefig('upsampling_zoom.png')
plt.show()


## 8. Evaluasi Kuantitatif: PSNR & SSIM

Untuk mengukur seberapa dekat hasil down-sample → up-sample dengan citra asli, kita
hitung **PSNR** (Peak Signal-to-Noise Ratio, dalam dB — semakin tinggi semakin mirip)
dan **SSIM** (Structural Similarity Index, 0–1 — semakin mendekati 1 semakin mirip
secara struktural) untuk setiap kombinasi metode down-sampling × up-sampling, pada
dua faktor skala (4x dan 8x).


In [ ]:
import pandas as pd

rows = []
for name, img in images.items():
    for factor in [4, 8]:
        for dmethod in ['max', 'average', 'median']:
            small = downsample(img, factor, dmethod)
            for umethod in ['nn', 'bilinear', 'bicubic']:
                big = upsample(small, factor, umethod)
                h, w = big.shape[:2]
                ref = img[:h, :w]
                p = psnr(ref, big, data_range=255)
                s = ssim(ref, big, channel_axis=2)
                rows.append({
                    'image': name, 'factor': factor,
                    'downsample': dmethod, 'upsample': umethod,
                    'PSNR_dB': round(p, 2), 'SSIM': round(s, 4)
                })

df = pd.DataFrame(rows)
df.to_csv('metrics_results.csv', index=False)
df


## 9. Ringkasan Rata-rata per Metode

Rata-rata PSNR & SSIM dikelompokkan per metode down-sampling dan up-sampling
(digabung dari semua citra dan faktor skala) untuk melihat kecenderungan umum.


In [ ]:
print('Rata-rata berdasarkan metode DOWN-SAMPLING:')
display(df.groupby('downsample')[['PSNR_dB', 'SSIM']].mean().round(3))

print('\nRata-rata berdasarkan metode UP-SAMPLING:')
display(df.groupby('upsample')[['PSNR_dB', 'SSIM']].mean().round(3))

print('\nRata-rata berdasarkan CITRA (menunjukkan pengaruh karakteristik citra):')
display(df.groupby('image')[['PSNR_dB', 'SSIM']].mean().round(3))


## 10. Grafik Perbandingan PSNR

Bar chart untuk memvisualisasikan rata-rata PSNR tiap kombinasi metode down-sampling
dan up-sampling.


In [ ]:
pivot = df.groupby(['downsample', 'upsample'])['PSNR_dB'].mean().unstack()
pivot = pivot[['nn', 'bilinear', 'bicubic']].loc[['max', 'average', 'median']]

pivot.plot(kind='bar', figsize=(8, 5))
plt.ylabel('Rata-rata PSNR (dB)')
plt.title('Rata-rata PSNR per kombinasi metode Down-Sampling x Up-Sampling')
plt.xticks(rotation=0)
plt.legend(title='Up-sampling')
plt.tight_layout()
plt.savefig('psnr_barchart.png')
plt.show()


## 11. Analisis & Kesimpulan

### Down Sampling
- **Max Pooling** secara konsisten menghasilkan PSNR/SSIM **terendah** di semua citra
  uji. Ini karena max pooling secara sistematis menggeser kecerahan citra ke atas
  (bias terang) dan sangat sensitif terhadap detail frekuensi tinggi, sehingga
  paling banyak kehilangan informasi warna/intensitas asli.
- **Average Pooling** dan **Median Pooling** menghasilkan hasil yang jauh lebih baik
  dan hampir setara satu sama lain pada citra dengan tekstur halus (astronaut, camera).
  Average pooling sedikit lebih unggul pada citra dengan gradasi warna halus,
  sedangkan **median pooling lebih unggul pada citra dengan pola tegas/frekuensi
  tinggi** (checkerboard) karena median tidak "mencampur" dua warna kontras menjadi
  abu-abu seperti yang dilakukan average.
- Pada citra **checkerboard**, seluruh metode menghasilkan PSNR yang jauh lebih
  rendah dibanding citra natural. Ini menunjukkan **efek aliasing**: pola berulang
  frekuensi tinggi sangat rentan terhadap distorsi ketika di-downsample tanpa
  low-pass filtering terlebih dahulu.
- Semakin besar faktor downsampling (8x vs 4x), semakin besar pula penurunan
  PSNR/SSIM untuk seluruh metode — sesuai ekspektasi karena semakin banyak
  informasi piksel yang dibuang.

### Up Sampling
- **Bicubic** secara konsisten memberikan PSNR & SSIM **tertinggi**, karena
  menggunakan interpolasi kubik dari 16 piksel tetangga sehingga transisi warna
  lebih halus dan mendekati citra asli.
- **Bilinear** berada di posisi kedua, sedikit di bawah bicubic, dengan biaya
  komputasi yang lebih ringan.
- **Nearest Neighbor (NN)** memberikan hasil **terendah** dan secara visual tampak
  "blocky" / berkotak-kotak, karena tidak melakukan interpolasi sama sekali — hanya
  menyalin nilai piksel tetangga terdekat. Namun NN adalah yang **tercepat** secara
  komputasi.

### Trade-off Umum
| Metode | Kualitas Visual | Kecepatan | Cocok untuk |
|---|---|---|---|
| Max Pool (down) | Rendah, bias terang | Cepat | Deteksi fitur/tepi terang |
| Average Pool (down) | Baik, halus | Cepat | Citra natural, mengurangi noise |
| Median Pool (down) | Baik, tahan noise impulsif | Sedang | Citra dengan tepi tegas/pola |
| NN (up) | Blocky | Sangat cepat | Preview cepat, real-time |
| Bilinear (up) | Halus, cukup tajam | Sedang | Kompromi umum (default banyak library) |
| Bicubic (up) | Paling halus & tajam | Lebih lambat | Kualitas visual/output akhir |

### Kesimpulan
Tidak ada satu metode yang "terbaik mutlak" untuk semua kasus — pemilihan metode
down-sampling dan up-sampling bergantung pada **karakteristik citra** (tekstur halus vs
pola frekuensi tinggi) dan **kebutuhan aplikasi** (kecepatan vs kualitas visual).
Secara umum, kombinasi **Average/Median Pooling** untuk down-sampling dan
**Bicubic** untuk up-sampling memberikan hasil kuantitatif dan visual terbaik pada
percobaan ini, sedangkan **Max Pooling** dan **Nearest Neighbor** paling cocok
ketika kecepatan komputasi lebih diprioritaskan daripada kualitas visual.
